# 🔬 Experiências: Modelos SOTA & Interpretabilidade

**Branch 2 — Laura** | Classificação de imagens de CPRE com Deep Learning

---

## 🗺️ Guia de Utilização

Algumas células têm código comentado com `#`. Isso é intencional:
o código que treina e avalia o modelo precisa dos **DataLoaders** da Branch 1.

**Passo a passo quando tiveres os DataLoaders:**
1. Corre a **Célula 1** (Setup) — sempre
2. Corre a **Célula 2** (Imports) — sempre
3. Na **Célula 3**, substitui o comentário pelo import da função da tua colega
4. A partir daí, **descomenta e corre** as células na ordem

## ⚙️ Célula 1 — Setup do Ambiente no Colab

> **Corre sempre esta célula primeiro.** Instala as bibliotecas necessárias.

In [ ]:
!pip install wandb -qU
!pip install grad-cam -q
import wandb
wandb.login()  # Cola a tua API Key do WandB quando pedido

## 📦 Célula 2 — Imports e Device

> **Corre sempre esta célula.** Importa todos os módulos do projeto.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

from models import get_model
from train_wandb import TrainerWandb
from metrics import MetricsEvaluator
from gradcam_utils import InterpretabilityTools, get_target_layers_for_model
from noise_robustness import evaluate_robustness, print_robustness_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device em uso:', device)

CLASS_NAMES = ['Biliary_Leaks', 'Lithiasis', 'Stricture', 'Normal']
evaluator = MetricsEvaluator(CLASS_NAMES)

## 🗂️ Célula 3 — DataLoaders

> **⚠️ AÇÃO NECESSÁRIA:** Quando receberes o código da Branch 1:
> 1. Apaga a linha `print("Aguardando...")`
> 2. Descomenta as duas linhas do `import` e da chamada à função
> 3. Garante que o ficheiro `data_pipeline.py` da tua colega está na mesma pasta

In [ ]:
# ⚠️ DESCOMENTA ESTAS LINHAS QUANDO TIVERES OS DATALOADERS DA BRANCH 1:
# from data_pipeline import get_dataloaders
# train_loader, val_loader, test_loader = get_dataloaders(batch_size=32)

print("Aguardando DataLoaders da Branch 1...")

## 🧠 Célula 4 — Configuração do Modelo e Treino

> **⚠️ AÇÃO NECESSÁRIA:** Descomenta as últimas 2 linhas (trainer + best_f1) quando tiveres os DataLoaders.
>
> Troca `model_name` para comparar os diferentes modelos:
> - `'densenet121'` — 🏥 **Recomendado para raio-X** (baseado no paper CheXNet, Stanford)
> - `'efficientnet_v2_s'` — ⚡ Rápido e eficiente, ótimo ponto de partida
> - `'resnet50'` — 🏛️ Clássico e muito estável
> - `'vit_b_16'` — 🤖 Vision Transformer, gera Attention Maps nativos

In [ ]:
config = {
    'epochs': 50,
    'batch_size': 32,
    'learning_rate': 1e-4,
    'model_name': 'densenet121',
    'transfer_learning': False  # False = fine-tuning completo (mais potente)
}

# Se o dataset estiver muito desbalanceado, usar class_weights:
# class_weights = torch.tensor([w_leaks, w_lithiasis, w_stricture, w_normal]).to(device)
# criterion = nn.CrossEntropyLoss(weight=class_weights)
criterion = nn.CrossEntropyLoss()

model = get_model(config['model_name'], num_classes=4, feature_extracting=config['transfer_learning'])
optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
print(f"Modelo pronto: {config['model_name']}")

# ⚠️ DESCOMENTA QUANDO TIVERES OS DATALOADERS:
# trainer = TrainerWandb(model, train_loader, val_loader, criterion, optimizer, device, CLASS_NAMES)
# best_f1 = trainer.train_and_evaluate(config, epochs=config['epochs'])

## 📊 Célula 5 — Avaliação no Conjunto de Teste

> **⚠️ AÇÃO NECESSÁRIA:** Descomenta TUDO nesta célula após o treino estar concluído.
> Esta célula gera as 3 métricas obrigatórias do professor:
> - F1-Score Macro (o que precisas de bater > 0.738)
> - Matriz de Confusão
> - Curvas AUC-ROC por classe

In [ ]:
# ⚠️ DESCOMENTA TUDO ABAIXO APÓS O TREINO:

# model.load_state_dict(torch.load('checkpoints/best_model.pth'))
# model.eval()

# all_preds, all_labels = [], []
# with torch.no_grad():
#     for inputs, labels in test_loader:
#         probs = torch.softmax(model(inputs.to(device)), dim=1)
#         all_preds.append(probs.cpu())
#         all_labels.append(labels)
# all_preds = torch.cat(all_preds)
# all_labels = torch.cat(all_labels)

# results = evaluator.evaluate(all_labels, all_preds)
# print(f"\nF1-Score Macro obtido: {results['f1_macro']:.4f}  (baseline: 0.738)")
# print(results['report'])

# evaluator.plot_confusion_matrix(
#     results['y_true'], results['y_pred_labels'],
#     title=f"Matriz de Confusão — {config['model_name']}",
#     save_path="confusion_matrix.png"
# )

# evaluator.plot_roc_curves(
#     all_labels, all_preds,
#     title=f"Curvas AUC-ROC — {config['model_name']}",
#     save_path="roc_curves.png"
# )

print("Célula de avaliação pronta — descomenta após o treino.")

## 🛡️ Célula 6 — Teste de Robustez ao Ruído

> **⚠️ AÇÃO NECESSÁRIA:** Descomenta após o treino.
> Testa se o modelo aguenta variações comuns em fluoroscopia:
> ruído Gaussiano, variações de contraste, artefactos de pixels.

In [ ]:
# ⚠️ DESCOMENTA APÓS O TREINO:

# robustness_results = evaluate_robustness(
#     model=model,
#     val_loader=val_loader,
#     evaluator=evaluator,
#     device=device
# )
# print_robustness_report(robustness_results)

print("Teste de robustez pronto — descomenta após o treino.")

## 🔥 Célula 7 — Interpretabilidade: Grad-CAM

> **⚠️ AÇÃO NECESSÁRIA:** Descomenta após o treino.
> Requisito **obrigatório** do professor. Gera os heatmaps que mostram
> para onde o modelo está a olhar na imagem de CPRE.
>
> ✅ Verifica se o foco está nos **ductos biliares** (zona correta)
> e não em artefactos externos ou marcadores.

In [ ]:
# ⚠️ DESCOMENTA APÓS O TREINO:

# target_layers = get_target_layers_for_model(model, config['model_name'])
# interpreter = InterpretabilityTools(model, target_layers)

# inputs, labels = next(iter(val_loader))
# img_tensor = inputs[0:1].to(device)
# img_rgb = inputs[0].permute(1, 2, 0).cpu().numpy()
# img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() - img_rgb.min())
# img_rgb = img_rgb.astype(np.float32)

# heatmap_img = interpreter.generate_gradcam(
#     img_tensor, img_rgb,
#     target_category=labels[0].item(),
#     save_path=f"gradcam_{CLASS_NAMES[labels[0].item()]}.png"
# )

# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# axes[0].imshow(img_rgb, cmap='gray')
# axes[0].set_title('Imagem Original')
# axes[0].axis('off')
# axes[1].imshow(heatmap_img)
# axes[1].set_title(f'Grad-CAM: {CLASS_NAMES[labels[0].item()]}')
# axes[1].axis('off')
# plt.tight_layout()
# plt.show()

print("Grad-CAM pronto — descomenta após o treino.")